# Option Pricing Model (OPM) - Project Horizon Full Valuation

This notebook is the working build for the contract-faithful valuation of the Project Horizon Option Agreement from Olympus BV's perspective.

## Valuation objective
- Primary output: integrated net contract fair value, not separate call and put values priced independently and then netted
- DLOM: retained in the primary valuation
- Regulatory lapse risk: discussed qualitatively and not quantified in the primary fair value
- Clean-value convention: leakage, dividends, and SPA claim adjustments remain outside the primary model unless separately modeled

## Contract features captured in this build
- Call exercise window: after Year 3 to Year 5
- Put exercise window: after Year 3.5 to Year 6
- One Live Notice: overlap interaction between call and put rights
- Partial and multiple call exercise: where contractually available
- Residual-share put logic: Titan's put applies only to shares still held at the exercise date
- Strike mechanics: SPA price plus Applicable Coupon accrued to option completion

## Explicit modeling assumptions
- Windowed-American rights are approximated on a monthly Bermudan grid
- Base case assumes partial call exercise is available; a sensitivity shows the full-block-only case
- If both parties would exercise in the same modeled time bucket, the base case applies a neutral 50/50 notice-priority split
- Completion lag is set to zero in the primary valuation as a simplifying assumption: for valuation purposes, option exercise notice and option completion are assumed to occur on the same date
- All reported runs use 1,000,000 simulations

## Build notes
- The zero completion-lag assumption is about the gap between option notice and option completion under the Option Agreement; it is not about whether the original transaction or PPA process remained open after the valuation date
- Main output should be valuation plus economically meaningful sensitivities, not simulation-count sensitivities
- External-circulation discipline: every contract input should cite the exact clause or schedule reference, every external market input should cite provider and date, and every SPA-derived input should cite the exact SPA paragraph before issue
- Any row still marked as pending in the source register below should be treated as not yet fully cleared for external circulation

## 1. Inputs

### Step 1A. Input Provenance and Citation Control

Before any number is relied on externally, each input should be tagged as one of three types: contract-derived, SPA or transaction-document-derived, or external market data. For client, adviser, auditor, or bank circulation, each row should carry an exact citation.

If an SPA-derived row does not yet show the exact SPA paragraph reference, that row should be treated as still open.

| Input | Base value | Type | Source / citation requirement | Status |
|---|---|---|---|---|
| `TOTAL_EQUITY_USD` / `SPA_price` | USD 3.2bn / USD 577.19 per share | SPA-derived | Exact SPA paragraph citation required. The SPA file is not present in the current workspace, so this citation remains outstanding. | Pending |
| `INITIAL_TITAN_PCT` / `OPTION_SHARES` | 25% / 1,386,020 shares | Contract-derived | Option Agreement Schedule 3 definition of `Option Shares`; Titan warranty at Clause 13.2.2 | Complete |
| `CALL_COUPONS` / `PUT_COUPONS` | Per code below | Contract-derived | Option Agreement Schedule 3 definition of `Applicable Coupon` | Complete |
| `call_start`, `call_end`, `put_start`, `put_end` | 3.0y-5.0y / 3.5y-6.0y | Contract-derived | Clause 4.1 and Schedule 3 definitions of `Call Option Period`, `Put Option Period`, and `Overlap Period` | Complete |
| `MIN_CALL_TRANCHE_PCT` / `RESIDUAL_FLOOR_PCT` | 5% / 7% | Contract-derived | Clause 2.2 minimum-tranche and residual-floor restrictions | Complete |
| `COMPLETION_LAG_YEARS` | 0.0 years | Modeling assumption | Schedule 3 definition of `Option Completion Date` contemplates 20 business days; primary valuation sets lag to zero as an explicit simplifying assumption | Complete as assumption |
| `OLYMPUS_PRIORITY_IN_OVERLAP` | 50% | Modeling assumption | Clause 8.1 gives earlier notice priority; 50/50 is the model's neutral same-bucket approximation | Complete as assumption |
| `asset_volatility` | 20.0% | External market data / judgment | Selected KO bottler peer asset-volatility analysis, Capital IQ, as of 31-December-2025 | Complete |
| `risk_free_rate` | 4.929% | External market data | Bloomberg USD risk-free input, as of 31-December-2025 | Complete |

In [22]:
import numpy as np

# ============================================================
# INPUT PROVENANCE / CITATION CONTROL
# ============================================================
# External-circulation rule:
# - contract-derived inputs cite the signed Option Agreement by clause / schedule;
# - external market inputs cite provider and date;
# - SPA-derived inputs cite the exact SPA paragraph before issue.
INPUT_SOURCE_REGISTER = {
    "TOTAL_EQUITY_USD / SPA_price": {
        "citation": "SPA-derived transaction input. Exact SPA paragraph citation remains required before external circulation; the SPA file is not present in the current workspace.",
        "status": "pending",
    },
    "INITIAL_TITAN_PCT / OPTION_SHARES": {
        "citation": "Option Agreement Schedule 3 definition of 'Option Shares'; Titan warranty at Clause 13.2.2.",
        "status": "complete",
    },
    "CALL_COUPONS / PUT_COUPONS": {
        "citation": "Option Agreement Schedule 3 definition of 'Applicable Coupon'.",
        "status": "complete",
    },
    "call_start / call_end / put_start / put_end": {
        "citation": "Clause 4.1 together with Schedule 3 definitions of 'Call Option Period', 'Put Option Period' and 'Overlap Period'.",
        "status": "complete",
    },
    "MIN_CALL_TRANCHE_PCT / RESIDUAL_FLOOR_PCT": {
        "citation": "Clause 2.2 minimum-tranche and residual-floor restrictions.",
        "status": "complete",
    },
    "COMPLETION_LAG_YEARS": {
        "citation": "Schedule 3 definition of 'Option Completion Date'; primary valuation sets lag to zero as an explicit simplifying assumption.",
        "status": "complete",
    },
    "OLYMPUS_PRIORITY_IN_OVERLAP": {
        "citation": "Clause 8.1 gives earlier notice priority; 50/50 is the model's neutral same-bucket approximation.",
        "status": "complete",
    },
    "asset_volatility": {
        "citation": "Selected KO bottler peer asset-volatility analysis, Capital IQ, as of 31-Dec-2025.",
        "status": "complete",
    },
    "risk_free_rate": {
        "citation": "Bloomberg USD risk-free input, as of 31-Dec-2025.",
        "status": "complete",
    },
}

# ============================================================
# CONTRACT TERMS / PRIMARY MODELING ASSUMPTIONS
# ============================================================
# SPA-derived reference equity value.
# Source required before external issue: exact SPA paragraph citation.
TOTAL_EQUITY_USD = 3_200_000_000

# Source: Option Agreement Schedule 3 definition of "Option Shares";
# corroborated by Titan warranty at Clause 13.2.2.
INITIAL_TITAN_PCT = 25
OPTION_SHARES = 1_386_020
TOTAL_SHARES = OPTION_SHARES / (INITIAL_TITAN_PCT / 100)
SPA_price = TOTAL_EQUITY_USD / TOTAL_SHARES

# Source: Option Agreement Schedule 3 definition of "Applicable Coupon".
CALL_COUPONS = np.array([0.0275, 0.0300, 0.0300, 0.0420, 0.0420], dtype=float)
PUT_COUPONS = np.array([0.0275, 0.0300, 0.0300, 0.0410, 0.0410, 0.0410], dtype=float)

# Source: Clause 4.1 together with Schedule 3 definitions of
# "Call Option Period", "Put Option Period" and "Overlap Period".
call_start, call_end = 3.0, 5.0
put_start, put_end = 3.5, 6.0

# Source: Clause 2.2 minimum-tranche and residual-floor restrictions.
MIN_CALL_TRANCHE_PCT = 5
RESIDUAL_FLOOR_PCT = 7
STATE_PCTS = np.array([0, 10, 15, 20, 25], dtype=int)

# Contract reference: Schedule 3 definition of "Option Completion Date".
# Primary simplifying assumption for this client-facing build:
# notice date and option completion date are aligned for valuation purposes.
COMPLETION_LAG_YEARS = 0.0

# Working base case: partial call exercise is available.
ALLOW_PARTIAL_CALLS = True

# Contract reference: Clause 8.1 gives earlier notice priority.
# Modeling simplification: 50/50 if both parties would act in the same monthly bucket.
OLYMPUS_PRIORITY_IN_OVERLAP = 0.50

# ============================================================
# MARKET / ECONOMIC ASSUMPTIONS
# ============================================================
# Source: selected KO bottler peer asset-volatility analysis,
# Capital IQ, as of 31-Dec-2025, rounded to 20.0%.
asset_volatility = 0.20

# Source: Bloomberg USD risk-free input, as of 31-Dec-2025.
risk_free_rate = 0.04929

# ============================================================
# SIMULATION SETTINGS
# ============================================================
# User-directed policy: all reported runs use 1,000,000 simulations.
simulations = 1_000_000
steps_per_year = 12
dt = 1 / steps_per_year
total_years = 6
total_steps = int(total_years / dt) + 1
time_grid = np.arange(total_steps) * dt

pending_source_items = [
    name for name, meta in INPUT_SOURCE_REGISTER.items() if meta["status"] != "complete"
]

np.random.seed(42)
print(f"SPA price per share:                      USD {SPA_price:,.2f}")
print(f"Initial option block value:               USD {SPA_price * OPTION_SHARES:,.0f}")
print(f"Base remaining state:                     {INITIAL_TITAN_PCT}% of total share capital")
print("Completion lag assumption:                0.000 years")
print("                                           Simplifying assumption: notice date = completion date")
print(f"Partial call assumption:                  {'Enabled' if ALLOW_PARTIAL_CALLS else 'Disabled'}")
print(f"Overlap priority to Olympus:              {OLYMPUS_PRIORITY_IN_OVERLAP:.0%}")
print(f"Asset volatility:                         {asset_volatility:.2%}  (Capital IQ, 31-Dec-2025)")
print(f"Risk-free rate:                           {risk_free_rate:.3%}  (Bloomberg, 31-Dec-2025)")
print(f"Simulations used in every reported run:   {simulations:,}")
if pending_source_items:
    print("Source-control exceptions:")
    for name in pending_source_items:
        print(f"- {name}: {INPUT_SOURCE_REGISTER[name]['citation']}")
else:
    print("Source-control exceptions:                None")

SPA price per share:                      USD 577.19
Initial option block value:               USD 800,000,000
Base remaining state:                     25% of total share capital
Completion lag assumption:                0.000 years
                                           Simplifying assumption: notice date = completion date
                                           This is not an assumption about unfinished SPA close or PPA timing
Partial call assumption:                  Enabled
Overlap priority to Olympus:              50%
Asset volatility:                         20.00%  (Capital IQ, 31-Dec-2025)
Risk-free rate:                           4.929%  (Bloomberg, 31-Dec-2025)
Simulations used in every reported run:   1,000,000
Source-control exceptions:
- TOTAL_EQUITY_USD / SPA_price: SPA-derived transaction input. Exact SPA paragraph citation remains required before external circulation; the SPA file is not present in the current workspace.


## 2. Contract Strike Paths

The agreement defines purchase price by reference to the SPA price plus the Applicable Coupon accrued to option completion.

The key point on completion lag is the following:
- It is the gap between **option exercise notice** and **option completion** under the Option Agreement.
- It is **not** the gap between the valuation date and the original SPA close.
- It is **not** an assumption that the transaction or PPA remained unfinished after the valuation date.

In principle, if option completion occurs after notice, the coupon continues to accrue through that completion date. In this client-facing build, that lag is set to **zero** as a practical simplifying assumption. That means the model assumes, for valuation purposes, that exercise and completion occur on the same date, so no additional coupon accrues after exercise notice in the primary valuation.

In [23]:
def accrued_price(base_price, coupons, t_years):
    """Accrue SPA price by the contractual Applicable Coupon schedule."""
    acc = base_price
    remaining = max(float(t_years), 0.0)
    for rate in coupons:
        if remaining <= 0:
            break
        year_slice = min(1.0, remaining)
        acc *= (1 + rate) ** year_slice
        remaining -= year_slice
    if remaining > 0:
        acc *= (1 + coupons[-1]) ** remaining
    return acc


def build_notice_strike_path(base_price, coupons, time_grid, completion_lag_years, rf):
    """
    Build strike paths to expected option completion.

    With completion_lag_years = 0.0, this simplifies to notice date = completion date.
    """
    strike_completion = np.zeros_like(time_grid)
    strike_pv = np.zeros_like(time_grid)
    completion_grid = np.zeros_like(time_grid)

    for idx, t_notice in enumerate(time_grid):
        t_completion = t_notice + completion_lag_years
        strike_completion[idx] = accrued_price(base_price, coupons, t_completion)
        strike_pv[idx] = np.exp(-rf * completion_lag_years) * strike_completion[idx]
        completion_grid[idx] = t_completion

    return strike_completion, strike_pv, completion_grid


call_strike_completion, call_strike_pv, call_completion_grid = build_notice_strike_path(
    SPA_price, CALL_COUPONS, time_grid, COMPLETION_LAG_YEARS, risk_free_rate
)
put_strike_completion, put_strike_pv, put_completion_grid = build_notice_strike_path(
    SPA_price, PUT_COUPONS, time_grid, COMPLETION_LAG_YEARS, risk_free_rate
)

for yr in [3.0, 3.5, 5.0, 6.0]:
    idx = int(round(yr / dt))
    if idx < total_steps:
        print(
            f"Exercise at Year {yr:>3.1f} | "
            f"Call strike used: {call_strike_completion[idx]:.4f} | "
            f"Put strike used: {put_strike_completion[idx]:.4f}"
        )

Exercise at Year 3.0 | Call strike used: 629.1827 | Put strike used: 629.1827
Exercise at Year 3.5 | Call strike used: 642.2596 | Put strike used: 641.9514
Exercise at Year 5.0 | Call strike used: 683.1439 | Put strike used: 681.8333
Exercise at Year 6.0 | Call strike used: 711.8360 | Put strike used: 709.7885


## 3. Simulate Equity Value Paths

Under the risk-neutral measure the equity drifts at the risk-free rate.
The same simulated paths are then fed into the integrated contract engine.

In [24]:
def simulate_gbm_paths(n_paths, total_steps, dt, spot0, rf, vol, seed=42):
    rng = np.random.default_rng(seed)
    paths = np.zeros((n_paths, total_steps), dtype=float)
    paths[:, 0] = spot0

    drift = (rf - 0.5 * vol**2) * dt
    diffusion = vol * np.sqrt(dt)

    for step in range(1, total_steps):
        z = rng.standard_normal(n_paths)
        paths[:, step] = paths[:, step - 1] * np.exp(drift + diffusion * z)

    return paths


paths = simulate_gbm_paths(
    n_paths=simulations,
    total_steps=total_steps,
    dt=dt,
    spot0=SPA_price,
    rf=risk_free_rate,
    vol=asset_volatility,
    seed=42,
)

print(f"Simulated {simulations:,} paths x {total_steps} steps.")
print(
    f"Mean equity at Year 6: {paths[:, -1].mean():.2f} "
    f"(risk-neutral expectation ~ {SPA_price * np.exp(risk_free_rate * total_years):.2f})"
)

Simulated 1,000,000 paths x 73 steps.
Mean equity at Year 6: 776.08 (risk-neutral expectation ~ 775.82)


## 4. Integrated Contract Engine

The FULL model values the Option Agreement as one state-dependent contract rather than as independent call and put legs.

### High-level methodological structure

- Time grid: monthly Bermudan exercise grid
- Ownership state grid: Titan's remaining shareholding under the agreement is discretised to 25%, 20%, 15%, 10%, or 0%
- Decision rule at each date: compare waiting, Olympus BV's admissible call actions, and Titan's right to put all remaining shares
- Overlap treatment: if both sides would exercise in the same modeled bucket, use the explicit notice-priority approximation described above
- Numerical technique: Longstaff-Schwartz regressions are fit separately by remaining-state to estimate continuation value

### What the remaining-state grid means

- The contract starts today in the 25% state. That is why the 25% line later in the notebook equals the base-case contract value.
- The 20%, 15%, and 10% states are not averages and are not assumed starting positions. They are conditional states the model could move into if earlier call exercises had already reduced Titan's holding.
- The 0% state means the contract has been exhausted because Titan no longer holds any shares under the agreement.
- The model does not assume that all 25% must transact together in the base case. From the 25% state, Olympus BV may wait or call to 20%, 15%, 10%, or 0%, subject to the contractual tranche and residual-floor rules.
- The remaining-state table later in the notebook is therefore a diagnostic by state, not a probability-weighted average across states.

### Financial meaning of exercise in this model

- Exercising a call on a tranche does **not** mean Olympus BV loses the upside on that tranche. It means Olympus BV converts option value on that tranche into owned equity.
- In valuation terms, the immediate exercise value of a called tranche is the current share value less the contractual strike on that tranche.
- After exercise, the model continues only on the **remaining contract state** because continuing to model option upside on shares already acquired would double count value.
- What is given up by exercising is not the economic upside of the acquired shares; what is given up is the **optionality to wait** on that same tranche.
- Partial exercise can therefore be rational in the model if it reduces future put exposure while preserving optionality on the residual shares.

### Why the model uses 5% steps rather than any arbitrary percentage

- The current build does **not** solve for a continuous optimal tranche such as 7.518%.
- The action set is discretised because the contract itself imposes a **minimum 5% call tranche** and a **7% residual floor unless the position goes to zero**.
- To keep the state space tractable and aligned with the contractual mechanics, the model approximates the remaining ownership with the discrete states 25%, 20%, 15%, 10%, and 0%.
- So the 5% increments are not random. They are a modeling approximation anchored to the contractual minimum-tranche rule.

### Could the exact call percentage be solved more precisely?

- Yes, in principle.
- If Titan currently holds `c%` at a given month, Olympus BV's admissible call size `x%` is any amount such that `x >= 5` and Titan is left with either `c - x = 0` or `c - x >= 7`.
- So, for example, from 25% the exact contract-feasible partial-call interval is 5%-18%, plus 25% to zero; from 20% it is 5%-13%, plus 20% to zero; from 15% it is 5%-8%, plus 15% to zero; and from 10% only 10%-to-zero is feasible.
- A fully rational Olympus BV would therefore not ask "5% or 25%?" in the abstract. It would choose the admissible call size that maximises immediate value on the called tranche plus continuation value on the residual contract.
- The current notebook does not yet solve that continuous optimisation exactly. It solves a coarser discrete approximation of that problem.

### Does the overlap become game theory?

- Yes. Once Titan is also modelled as fully rational during the overlap period, the problem becomes a two-party stochastic game rather than a single-party stopping problem.
- From Olympus BV's perspective, Olympus tries to maximise contract value and Titan tries to minimise it, so the option contract is naturally close to a zero-sum game.
- The current build is not yet that full game-theoretic solution. It values the contract from Olympus BV's perspective with a discrete Olympus action menu and a neutral same-bucket notice-priority approximation for overlap.
- A more exact next-step model would therefore combine a finer or continuous call-size search with explicit Olympus-versus-Titan min/max or sequential notice-priority logic.

### Why the current coarse grid may still be informative

- Under the current calibration, contract value is close to linear in the remaining stake.
- That is why partial-call flexibility adds only modest value relative to the full-block-only sensitivity in the current results.
- In that setting, the exact interior optimum call size often matters less than whether Olympus waits, reduces the block, or takes Titan out completely.

### What the current build is and is not

- It is a discrete state-grid approximation of the linked contract.
- It is not a standalone call valuation minus a standalone put valuation.
- It is not a full game-theoretic stopping model.
- It does not take an average of 25%, 20%, 15%, 10%, and 0% states to arrive at the base-case value.
- It does not optimise over arbitrary fractional tranches between those grid points.

### How to interpret the reported value

| Item | Sign to Olympus BV | Included in current FULL model? | Reported as a separate standalone value? |
|---|---|---|---|
| Call rights | Positive | Yes | No |
| Put obligations | Negative | Yes | No |
| Interaction between the two rights | Can increase or decrease net value | Yes | No |
| Reported fair value | Net value of the linked contract from Olympus BV's perspective | Yes | Yes |

So the economic intuition is still "long call, short put" from Olympus BV's perspective, but the reported number in this FULL notebook is **not** a standalone call value minus a standalone put value priced independently. It is the direct value of the **interconnected agreement as a whole**.

In [ ]:
def shares_for_pct(pct):
    return TOTAL_SHARES * (pct / 100.0)


def allowed_call_targets(current_pct, allow_partial_calls):
    if current_pct == 0:
        return []
    if not allow_partial_calls:
        return [0]
    return [
        target for target in STATE_PCTS
        if target < current_pct
        and (current_pct - target) >= MIN_CALL_TRANCHE_PCT
        and (target == 0 or target >= RESIDUAL_FLOOR_PCT)
    ]


def basis_functions(spot, spot0):
    x = spot / spot0
    return np.column_stack([np.ones_like(x), x, x**2, x**3])


def fit_continuation_coeffs(spot, discounted_next_values, spot0):
    X = basis_functions(spot, spot0)
    coeffs, _, _, _ = np.linalg.lstsq(X, discounted_next_values, rcond=None)
    return coeffs


def predict_continuation(spot, coeffs, spot0):
    return basis_functions(spot, spot0) @ coeffs


def in_window(t, start, end, tol=1e-10):
    return (t + tol) >= start and (t - tol) <= end


def value_integrated_contract(
    paths,
    time_grid,
    dt,
    rf,
    call_strike_pv,
    put_strike_pv,
    allow_partial_calls,
    olympus_priority,
    state_pcts,
    spot0,
):
    n_paths = paths.shape[0]
    one_step_discount = np.exp(-rf * dt)
    next_values_by_state = {int(pct): np.zeros(n_paths, dtype=float) for pct in state_pcts}

    for step in range(len(time_grid) - 1, -1, -1):
        current_spot = paths[:, step]
        discounted_next_by_state = {
            pct: one_step_discount * next_values_by_state[pct]
            for pct in next_values_by_state
        }
        continuation_coeffs = {
            pct: fit_continuation_coeffs(current_spot, discounted_next_by_state[pct], spot0)
            for pct in discounted_next_by_state
        }
        continuation_estimates = {
            pct: predict_continuation(current_spot, continuation_coeffs[pct], spot0)
            for pct in continuation_coeffs
        }

        current_values_by_state = {0: np.zeros(n_paths, dtype=float)}
        t = time_grid[step]
        call_live = in_window(t, call_start, call_end)
        put_live = in_window(t, put_start, put_end)

        call_intrinsic = np.maximum(current_spot - call_strike_pv[step], 0.0) if call_live else None
        put_intrinsic = np.maximum(put_strike_pv[step] - current_spot, 0.0) if put_live else None

        for current_pct in [10, 15, 20, 25]:
            wait_est = continuation_estimates[current_pct]
            wait_real = discounted_next_by_state[current_pct]

            best_call_est = wait_est.copy()
            best_call_real = wait_real.copy()

            if call_live:
                for target_pct in allowed_call_targets(current_pct, allow_partial_calls):
                    called_shares = shares_for_pct(current_pct - target_pct)

                    # Immediate call value is booked only on the tranche actually acquired.
                    # Continuation then values only the residual contract state so the model
                    # does not double count upside on shares already purchased.
                    exercise_now = call_intrinsic * called_shares
                    candidate_est = exercise_now + continuation_estimates[target_pct]
                    better = candidate_est > best_call_est
                    if np.any(better):
                        candidate_real = exercise_now + discounted_next_by_state[target_pct]
                        best_call_est = np.where(better, candidate_est, best_call_est)
                        best_call_real = np.where(better, candidate_real, best_call_real)

            if put_live:
                # Titan's put, when live, is tested on all shares remaining in the current state.
                put_real = -(put_intrinsic * shares_for_pct(current_pct))
            else:
                put_real = np.zeros(n_paths, dtype=float)

            if call_live and not put_live:
                choose_call = best_call_est > wait_est + 1e-12
                current_values = np.where(choose_call, best_call_real, wait_real)
            elif put_live and not call_live:
                choose_put = put_real < wait_est - 1e-12
                current_values = np.where(choose_put, put_real, wait_real)
            elif call_live and put_live:
                choose_call = best_call_est > wait_est + 1e-12
                choose_put = put_real < wait_est - 1e-12
                both = choose_call & choose_put

                current_values = wait_real.copy()
                current_values = np.where(choose_call & ~choose_put, best_call_real, current_values)
                current_values = np.where(choose_put & ~choose_call, put_real, current_values)
                if np.any(both):
                    mixed = olympus_priority * best_call_real + (1.0 - olympus_priority) * put_real
                    current_values = np.where(both, mixed, current_values)
            else:
                current_values = wait_real

            current_values_by_state[current_pct] = current_values

        next_values_by_state = current_values_by_state

    initial_paths = next_values_by_state[INITIAL_TITAN_PCT]
    state_means = {pct: next_values_by_state[pct].mean() for pct in next_values_by_state}
    return {
        "value_paths_total": initial_paths,
        "value_per_share": initial_paths.mean() / OPTION_SHARES,
        "value_total": initial_paths.mean(),
        "value_std_total": initial_paths.std(ddof=1),
        "state_values_total": state_means,
    }


def run_full_model_scenario(n_paths, vol, rf, allow_partial_calls, olympus_priority, seed=42):
    scenario_paths = simulate_gbm_paths(
        n_paths=n_paths,
        total_steps=total_steps,
        dt=dt,
        spot0=SPA_price,
        rf=rf,
        vol=vol,
        seed=seed,
    )
    _, scenario_call_pv, _ = build_notice_strike_path(
        SPA_price, CALL_COUPONS, time_grid, COMPLETION_LAG_YEARS, rf
    )
    _, scenario_put_pv, _ = build_notice_strike_path(
        SPA_price, PUT_COUPONS, time_grid, COMPLETION_LAG_YEARS, rf
    )
    return value_integrated_contract(
        paths=scenario_paths,
        time_grid=time_grid,
        dt=dt,
        rf=rf,
        call_strike_pv=scenario_call_pv,
        put_strike_pv=scenario_put_pv,
        allow_partial_calls=allow_partial_calls,
        olympus_priority=olympus_priority,
        state_pcts=STATE_PCTS,
        spot0=SPA_price,
    )

### 4A. How Exercise Is Applied Inside the Monte Carlo

This is the key point: the 25% / 20% / 15% / 10% / 0% logic is applied **inside** the backward-induction Monte Carlo, not after the simulation as a separate overlay.

For each monthly decision date, for each simulated path, and for each current remaining-state, the model does the following:

1. Compute the value of **waiting**.
2. If the call window is open, test every **admissible call target state** on the discrete state grid.
3. If the put window is open, test Titan's right to **put all shares remaining in the current state**.
4. Choose the action implied by the model's decision rule for that path, date, and current remaining-state.

For example, from the **25%** state in the base case:
- Olympus BV can wait;
- Olympus BV can call **5%** and move the contract to the **20%** state;
- Olympus BV can call **10%** and move to the **15%** state;
- Olympus BV can call **15%** and move to the **10%** state; or
- Olympus BV can call the full remaining **25%** and move to **0%**.

If the model chooses the 25% -> 20% action on a given path and month, the immediate payoff is calculated only on the **5% actually called**, and the remaining **20%** continues to be valued from the 20% state. If later the put window is open while the contract is in the 20% state, Titan's put is tested on **20%**, not on the original 25%.

From a financial valuation perspective, the important trade-off is this:
- exercising a called tranche does **not** erase the economic upside on that tranche, because Olympus BV then owns those shares;
- what disappears is the **optionality to wait** on that tranche;
- partial exercise can therefore be preferred if it removes some put exposure now but preserves timing flexibility on the residual contract.

The current model does not solve for arbitrary fractional calls such as **7.518%**. The admissible call sizes are restricted to the discrete grid implied by the current state approximation and the contract constraints.

In principle that richer problem is solvable. Olympus BV could be allowed to choose any contract-feasible call size, and Titan could be modelled as responding strategically during overlap. But that would no longer be the current coarse state-grid approximation; it would be a two-party stochastic game with a much finer control set.

So the state grid is a discrete way of carrying forward the remaining ownership after partial call exercise. It is not a weighted average of states, and it is not an assumption that the full 25% block always transacts together.

#### Illustrative state ladder

```text
Current state at valuation date: Titan remaining 25%

25% state
  -> wait and remain in 25%
  -> call 5%  and move to 20%
  -> call 10% and move to 15%
  -> call 15% and move to 10%
  -> call 25% and move to 0%

20% state
  -> wait and remain in 20%
  -> call 5%  and move to 15%
  -> call 10% and move to 10%
  -> call 20% and move to 0%

15% state
  -> wait and remain in 15%
  -> call 5%  and move to 10%
  -> call 15% and move to 0%

10% state
  -> wait and remain in 10%
  -> call 10% and move to 0%

Put rule when live
  -> Titan may put all shares remaining in the current state
  -> so if the contract is in the 20% state, the put is tested on 20%, not 25%
```

## 5. Base Integrated Valuation

Run the integrated contract engine from the initial 25% state and retain the pathwise discounted values for reporting, DLOM, and the fixed-policy precision summary.

In [26]:
full_result = value_integrated_contract(
    paths=paths,
    time_grid=time_grid,
    dt=dt,
    rf=risk_free_rate,
    call_strike_pv=call_strike_pv,
    put_strike_pv=put_strike_pv,
    allow_partial_calls=ALLOW_PARTIAL_CALLS,
    olympus_priority=OLYMPUS_PRIORITY_IN_OVERLAP,
    state_pcts=STATE_PCTS,
    spot0=SPA_price,
)

full_value_paths_total = full_result["value_paths_total"]
full_value_total = full_result["value_total"]
full_value_per_share = full_result["value_per_share"]
full_value_std_total = full_result["value_std_total"]
full_value_se_total = full_value_std_total / np.sqrt(simulations)
state_values_total = full_result["state_values_total"]

print(f"Integrated contract value (per initial option share): USD {full_value_per_share:,.4f}")
print(f"Integrated contract value (total):                   USD {full_value_total / 1e6:,.1f}M")
print(f"Monte Carlo standard error:                          USD {full_value_se_total / 1e6:,.2f}M")

Integrated contract value (per initial option share): USD 100.7363
Integrated contract value (total):                   USD 139.6M
Monte Carlo standard error:                          USD 0.22M


## 6. Summary Before DLOM

This is the clean integrated contract value from Olympus BV's perspective before any marketability haircut.

The remaining-state table below should be read as a diagnostic of separate conditional states:
- 25% = the current initial state at the valuation date
- 20%, 15%, and 10% = conditional reduced states after hypothetical prior call exercise
- 0% = exhausted contract

It is not a weighted average across states. The fact that value scales almost linearly across these rows is an output of the current implementation, not an assumption that all states are economically identical.

In [37]:
print("=" * 72)
print("Integrated contract valuation before DLOM")
print("=" * 72)
print(f"Base case total value:        USD {full_value_total / 1e6:>8.1f}M")
print(f"Base case per option share:   USD {full_value_per_share:>8.2f}")
print(f"95% confidence interval:      +/- USD {1.96 * full_value_se_total / 1e6:>5.2f}M")
print("-" * 72)
print("Action menu applied inside the valuation")
print("-" * 72)
for current_pct in [25, 20, 15, 10]:
    action_labels = []
    for target_pct in allowed_call_targets(current_pct, ALLOW_PARTIAL_CALLS):
        called_pct = current_pct - target_pct
        action_labels.append(f"call {called_pct}% -> {target_pct}% remaining")
    print(
        f"State {current_pct:>2}% | "
        f"admissible calls: {'; '.join(action_labels)} | "
        f"if put is live: put all remaining {current_pct}%"
    )
print("State  0% | contract exhausted")
print("-" * 72)
print("Financial interpretation of call exercise")
print("- exercising a called tranche converts option value on that tranche into owned equity")
print("- the model then continues only on the remaining contract state to avoid double counting shares already acquired")
print("- what is preserved by partial exercise is optionality on the residual contract, not upside on shares already bought")
print("- current build is rational within a discrete menu, but it does not yet optimize over every contract-feasible call size")
print("- a fuller overlap treatment would be a zero-sum stochastic game: Olympus maximizes and Titan minimizes Olympus value")
print("- because current calibration is close to linear in the remaining stake, exact interior call size is not currently the main driver of value")
print("- 5% increments are not random; they come from the discrete state grid anchored to the 5% minimum tranche rule")
print("- arbitrary fractions such as 7.518% are not part of the current approximation")
print("- in the current calibration, partial-call flexibility is present but is not the main driver of total value")
print("-" * 72)
print("Conditional contract values by Titan remaining-state")
print("Diagnostic only; not a probability-weighted average across states")
print("-" * 72)
for pct in [25, 20, 15, 10, 0]:
    total_value = state_values_total[pct]
    if pct > 0:
        per_remaining_share = total_value / shares_for_pct(pct)
        state_role = "current initial state" if pct == INITIAL_TITAN_PCT else "conditional reduced state"
        print(
            f"Titan remaining {pct:>2}% | "
            f"{state_role:<25} | "
            f"contract value {total_value / 1e6:>8.1f}M | "
            f"per remaining share {per_remaining_share:>8.2f}"
        )
    else:
        print("Titan remaining  0% | contract exhausted")
print("-" * 72)
print("Pathwise interpretation:")
print("If a path chooses 25% -> 20% at a given month, payoff is booked on the called 5% only,")
print("and the remaining contract continues from the 20% state. Any later put is then tested on 20%, not 25%.")

Integrated contract valuation before DLOM
Base case total value:        USD    139.6M
Base case per option share:   USD   100.74
95% confidence interval:      +/- USD  0.43M
------------------------------------------------------------------------
Action menu applied inside the valuation
------------------------------------------------------------------------
State 25% | admissible calls: call 25% -> 0% remaining; call 15% -> 10% remaining; call 10% -> 15% remaining; call 5% -> 20% remaining | if put is live: put all remaining 25%
State 20% | admissible calls: call 20% -> 0% remaining; call 10% -> 10% remaining; call 5% -> 15% remaining | if put is live: put all remaining 20%
State 15% | admissible calls: call 15% -> 0% remaining; call 5% -> 10% remaining | if put is live: put all remaining 15%
State 10% | admissible calls: call 10% -> 0% remaining | if put is live: put all remaining 10%
State  0% | contract exhausted
---------------------------------------------------------------------

## 7. DLOM and Primary Fair Value

Longstaff's lookback-put approach is retained as the marketability adjustment.
The primary reported value is the integrated clean contract value after applying the central DLOM estimate.

In [28]:
def longstaff_dlom(paths, rf, dt, spot0, restriction_years):
    idx = min(int(round(restriction_years / dt)), paths.shape[1] - 1)
    actual_horizon = idx * dt
    s_max = paths[:, : idx + 1].max(axis=1)
    s_T = paths[:, idx]
    lookback_payoff = s_max - s_T
    lookback_value = np.exp(-rf * actual_horizon) * lookback_payoff.mean()
    return {
        "restriction_years": actual_horizon,
        "lookback_value": lookback_value,
        "dlom_pct": lookback_value / spot0,
    }


dlom_terms = {T: longstaff_dlom(paths, risk_free_rate, dt, SPA_price, T) for T in [3.0, 4.0, 5.0]}
dlom_central = dlom_terms[4.0]["dlom_pct"]
full_value_total_dlom = full_value_total * (1 - dlom_central)
full_value_per_share_dlom = full_value_per_share * (1 - dlom_central)

print("DLOM via Longstaff lookback put")
print("-" * 72)
for T in [3.0, 4.0, 5.0]:
    term = dlom_terms[T]
    print(
        f"Restriction horizon {term['restriction_years']:.1f}yr | "
        f"Lookback value {term['lookback_value']:>7.3f} | "
        f"DLOM {term['dlom_pct']:.2%}"
    )
print("-" * 72)
print(f"Primary fair value after central DLOM: USD {full_value_total_dlom / 1e6:,.1f}M")
print(f"Primary fair value per option share:   USD {full_value_per_share_dlom:,.2f}")

DLOM via Longstaff lookback put
------------------------------------------------------------------------
Restriction horizon 3.0yr | Lookback value 107.947 | DLOM 18.70%
Restriction horizon 4.0yr | Lookback value 121.699 | DLOM 21.08%
Restriction horizon 5.0yr | Lookback value 132.735 | DLOM 23.00%
------------------------------------------------------------------------
Primary fair value after central DLOM: USD 110.2M
Primary fair value per option share:   USD 79.50


## 8. Precision and Sensitivities

All reported scenario runs below use 1,000,000 simulations.
The notebook reports observed precision at that fixed simulation policy and then shows market and contract sensitivities on the same path count. It does not present a separate simulation-count sensitivity table.

In [29]:
print("Simulation policy and observed precision")
print("-" * 72)
print(f"Fixed simulation count for every reported run: {simulations:,}")
print(f"Observed Monte Carlo standard error:          USD {full_value_se_total / 1e6:,.2f}M")
print(f"Observed 95% confidence interval:            +/- USD {1.96 * full_value_se_total / 1e6:,.2f}M")

Simulation policy and observed precision
------------------------------------------------------------------------
Fixed simulation count for every reported run: 1,000,000
Observed Monte Carlo standard error:          USD 0.22M
Observed 95% confidence interval:            +/- USD 0.43M


In [30]:
vol_range = [0.15, 0.20, 0.25, 0.30]
rf_range = [0.04, risk_free_rate, 0.06]
SIM_SENS = simulations

print("Integrated fair value to Olympus BV (USD millions, before DLOM)")
print(f"All reported scenarios below use {SIM_SENS:,} simulations.")
print(f"{'Asset Vol':>10} | " + " | ".join(f" rf={rf:.2%} " for rf in rf_range))
print("-" * (14 + 15 * len(rf_range)))

scenario_id = 0
base_value_m = full_value_total / 1e6
for vol in vol_range:
    row = []
    for rf in rf_range:
        is_base_case = vol == asset_volatility and abs(rf - risk_free_rate) < 1e-12
        if is_base_case:
            value_m = base_value_m
        else:
            scenario_id += 1
            result = run_full_model_scenario(
                n_paths=SIM_SENS,
                vol=vol,
                rf=rf,
                allow_partial_calls=ALLOW_PARTIAL_CALLS,
                olympus_priority=OLYMPUS_PRIORITY_IN_OVERLAP,
                seed=200 + scenario_id,
            )
            value_m = result["value_total"] / 1e6
        marker = " *" if is_base_case else "  "
        row.append(f"{value_m:>8.1f}{marker}")
    print(f"{vol:>9.0%} | " + " | ".join(row))

print("\nBase case marked with *; every reported scenario follows the fixed 1,000,000-path policy.")

Integrated fair value to Olympus BV (USD millions, before DLOM)
All reported scenarios below use 1,000,000 simulations.
 Asset Vol |  rf=4.00%  |  rf=4.93%  |  rf=6.00% 
-----------------------------------------------------------
      15% |    100.2   |    112.0   |    126.0  
      20% |    128.5   |    139.6 * |    152.1  
      25% |    157.4   |    167.1   |    179.5  
      30% |    185.2   |    194.7   |    206.2  

Base case marked with *; every reported scenario follows the fixed 1,000,000-path policy.


## 9. Contract-Specific Sensitivities and Scope

The remaining uncertainties in the FULL build are not generic option inputs; they are contract-specific modeling choices.
This section keeps the simulation count fixed at 1,000,000 and changes only those contract-specific assumptions while keeping regulatory lapse risk qualitative only.

In [31]:
SIM_STRUCT = simulations
STRUCTURAL_SEED = 777
structural_cases = [
    ("Base case: partial calls, 50/50 overlap", True, 0.50),
    ("Full-block only, 50/50 overlap", False, 0.50),
    ("Partial calls, Olympus 25% overlap priority", True, 0.25),
    ("Partial calls, Olympus 75% overlap priority", True, 0.75),
]

print("Structural sensitivity (USD millions, before DLOM)")
print(f"All reported cases below use {SIM_STRUCT:,} simulations.")
print("-" * 72)
for label, allow_partial, priority in structural_cases:
    is_base_case = allow_partial == ALLOW_PARTIAL_CALLS and abs(priority - OLYMPUS_PRIORITY_IN_OVERLAP) < 1e-12
    if is_base_case:
        value_total = full_value_total
    else:
        result = run_full_model_scenario(
            n_paths=SIM_STRUCT,
            vol=asset_volatility,
            rf=risk_free_rate,
            allow_partial_calls=allow_partial,
            olympus_priority=priority,
            seed=STRUCTURAL_SEED,
        )
        value_total = result["value_total"]
    print(f"{label:<44} {value_total / 1e6:>8.1f}M")

print("\nRegulatory lapse risk remains qualitative and is not deducted from the primary value.")

Structural sensitivity (USD millions, before DLOM)
All reported cases below use 1,000,000 simulations.
------------------------------------------------------------------------
Base case: partial calls, 50/50 overlap         139.6M
Full-block only, 50/50 overlap                  139.1M
Partial calls, Olympus 25% overlap priority     139.1M
Partial calls, Olympus 75% overlap priority     137.5M

Regulatory lapse risk remains qualitative and is not deducted from the primary value.


In [32]:
print("Sliding contract state map under the base-case partial-call assumption")
print("-" * 72)
for current_pct in [25, 20, 15, 10]:
    transitions = []
    for target_pct in allowed_call_targets(current_pct, True):
        called_pct = current_pct - target_pct
        transitions.append(f"call {called_pct}% -> {target_pct}% remaining")
    print(f"From {current_pct:>2}% remaining: " + "; ".join(transitions))

print("\nTitan's put rule: at any eligible put date, Titan may put all remaining shares only.")

Sliding contract state map under the base-case partial-call assumption
------------------------------------------------------------------------
From 25% remaining: call 25% -> 0% remaining; call 15% -> 10% remaining; call 10% -> 15% remaining; call 5% -> 20% remaining
From 20% remaining: call 20% -> 0% remaining; call 10% -> 10% remaining; call 5% -> 15% remaining
From 15% remaining: call 15% -> 0% remaining; call 5% -> 10% remaining
From 10% remaining: call 10% -> 0% remaining

Titan's put rule: at any eligible put date, Titan may put all remaining shares only.


In [33]:
print("=" * 72)
print("Primary reported output")
print("=" * 72)
print(f"Integrated clean value before DLOM: USD {full_value_total / 1e6:,.1f}M")
print(f"Integrated fair value after DLOM:   USD {full_value_total_dlom / 1e6:,.1f}M")
print(f"Per initial option share after DLOM: USD {full_value_per_share_dlom:,.2f}")
print()
print("Scope note:")
print("- regulatory lapse risk remains qualitative only and is not deducted from the primary value")
print("- leakage, dividends, and SPA claim adjustments remain outside the primary model")
print("- the original notebook is preserved separately as the economic proxy / first-draft model")
if pending_source_items:
    print("- source-control exception(s) remain outstanding before external issue:")
    for name in pending_source_items:
        print(f"  * {name}")
else:
    print("- source-control register complete")

Primary reported output
Integrated clean value before DLOM: USD 139.6M
Integrated fair value after DLOM:   USD 110.2M
Per initial option share after DLOM: USD 79.50

Scope note:
- regulatory lapse risk remains qualitative only and is not deducted from the primary value
- leakage, dividends, and SPA claim adjustments remain outside the primary model
- the original notebook is preserved separately as the economic proxy / first-draft model
- source-control exception(s) remain outstanding before external issue:
  * TOTAL_EQUITY_USD / SPA_price
